# Day 6: Per-Stock Fitting and Disagreement Detection
We fit a separate NH-HMM and Baseline HMM for each stock, map states to consistent labels, and detect days where the models disagree exactly on an event date.


In [1]:
import sys
sys.path.append('../')

import os
import pandas as pd
import numpy as np
from hmmlearn.hmm import GaussianHMM
from src.nonhomogeneous_hmm import NonHomogeneousGaussianHMM
from src.regime_utils import label_states_by_mean
import warnings
warnings.filterwarnings('ignore')


## 1. Load Data


In [2]:
returns_with_events = pd.read_parquet('../data/returns_with_events.parquet')
events_table = pd.read_parquet('../data/events_table.parquet')

# Get unique stock symbols (exclude NIFTY50)
symbols = [sym for sym in returns_with_events['symbol'].unique() if sym != '^NSEI']
print(f"Fitting models for {len(symbols)} stocks.")

os.makedirs('../data/per_stock_states', exist_ok=True)


Fitting models for 17 stocks.


## 2. Fit Models and Decode States for Each Stock


In [3]:
all_disagreements = []

for sym in symbols:
    # 1. Get stock data
    stock_data = returns_with_events[returns_with_events['symbol'] == sym].copy()
    X = np.column_stack([stock_data['log_return'].values, stock_data['volatility'].values])
    E = stock_data['E_t'].values.astype(bool)
    dates = stock_data['date'].values
    
    # 2. Fit Baseline HMM
    best_base_ll = -np.inf
    best_base = None
    for i in range(3):
        model = GaussianHMM(n_components=3, covariance_type="diag", n_iter=100, random_state=i)
        model.fit(X)
        ll = model.score(X)
        if ll > best_base_ll:
            best_base_ll = ll
            best_base = model
            
    base_states = best_base.predict(X)
    
    # 3. Fit NH-HMM
    nh = NonHomogeneousGaussianHMM(n_components=3, n_iter=100, n_restarts=3, random_state=42)
    nh.fit(X, E)
    nh_ll = nh.score(X, E)
    nh_states = nh.decode(X, E)
    
    # 4. Standardize Labels
    base_labels_map, base_std_map = label_states_by_mean(best_base.means_, best_base.covars_)
    nh_labels_map, nh_std_map = label_states_by_mean(nh.means_, nh.covars_)
    
    stock_data['baseline_state'] = [base_std_map[s] for s in base_states]
    stock_data['baseline_label'] = [base_labels_map[s] for s in base_states]
    stock_data['nh_state'] = [nh_std_map[s] for s in nh_states]
    stock_data['nh_label'] = [nh_labels_map[s] for s in nh_states]
    
    # 5. Save per-stock states
    out_cols = ['date', 'baseline_state', 'baseline_label', 'nh_state', 'nh_label']
    stock_data[out_cols].to_parquet(f'../data/per_stock_states/{sym}.parquet')
    
    # 6. Find disagreements on EVENT DATES
    A_event_norm = np.linalg.norm(nh.A_event_ - nh.A_normal_, 'fro')
    
    sym_events = events_table[events_table['symbol'] == sym]
    
    for _, ev in sym_events.iterrows():
        ev_date = ev['event_date']
        # Check if event_date is a trading day
        day_data = stock_data[stock_data['date'] == ev_date]
        if len(day_data) == 0:
            continue
            
        day_data = day_data.iloc[0]
        
        if day_data['baseline_state'] != day_data['nh_state']:
            disagreement = {
                'symbol': sym,
                'event_date': ev_date,
                'event_type': ev['event_type'],
                'baseline_state': day_data['baseline_state'],
                'baseline_label': day_data['baseline_label'],
                'nh_state': day_data['nh_state'],
                'nh_label': day_data['nh_label'],
                'baseline_ll': best_base_ll,
                'nh_ll': nh_ll,
                'A_event_norm': A_event_norm,
                'days_to_event_signed': 0  # It's exactly on the event date
            }
            all_disagreements.append(disagreement)

disagreement_df = pd.DataFrame(all_disagreements)
print(f"Total raw disagreements found on event dates: {len(disagreement_df)}")


Model is not converging.  Current: 12234.922895043472 is not greater than 12235.123973461054. Delta is -0.20107841758181166


Model is not converging.  Current: 12934.091547729293 is not greater than 12936.178165529585. Delta is -2.0866178002925153


Model is not converging.  Current: 12840.57823083796 is not greater than 12840.636903153538. Delta is -0.05867231557749619


Model is not converging.  Current: 12619.847391036163 is not greater than 12620.798863166023. Delta is -0.9514721298601216


Model is not converging.  Current: 12647.864790180984 is not greater than 12648.649155035102. Delta is -0.7843648541183939


Model is not converging.  Current: 12650.583216325413 is not greater than 12651.597204107973. Delta is -1.0139877825604344


Model is not converging.  Current: 12956.547683384802 is not greater than 12956.654909992325. Delta is -0.10722660752253432


Model is not converging.  Current: 12557.360948915875 is not greater than 12557.388656738816. Delta is -0.027707822941010818


Model is not converging.  Current: 12557.8502773877 is not greater than 12557.851315839509. Delta is -0.00103845180819917


Total raw disagreements found on event dates: 500


## 3. Quality Filters


In [4]:
if len(disagreement_df) > 0:
    # Filter 1: A_event_norm threshold
    threshold = 0.01
    filtered_df = disagreement_df[disagreement_df['A_event_norm'] >= threshold]
    
    if len(filtered_df) < 80:
        print(f"Only {len(filtered_df)} rows with threshold 0.01. Reducing to 0.005.")
        threshold = 0.005
        filtered_df = disagreement_df[disagreement_df['A_event_norm'] >= threshold]
        
    print(f"Rows after A_event_norm >= {threshold}: {len(filtered_df)}")
    
    # Filter 2: Drop stocks with < 1 disagreement rows
    counts = filtered_df['symbol'].value_counts()
    valid_stocks = counts[counts >= 1].index
    dropped_stocks = counts[counts < 1].index
    
    print(f"Stocks dropped due to <1 rows: {list(dropped_stocks)}")
    
    filtered_df = filtered_df[filtered_df['symbol'].isin(valid_stocks)]
    print(f"Rows after dropping sparse stocks: {len(filtered_df)}")
    
    # Filter 3: Cap at N=15 rows per stock
    filtered_df['ll_diff'] = np.abs(filtered_df['nh_ll'] - filtered_df['baseline_ll'])
    filtered_df = filtered_df.sort_values(['symbol', 'll_diff'], ascending=[True, False])
    filtered_df = filtered_df.groupby('symbol').head(15).reset_index(drop=True)
    filtered_df = filtered_df.drop(columns=['ll_diff'])
    print(f"Capped at 15 rows per stock max.")
        
    disagreement_df = filtered_df
    
print("Final disagreement count per stock and Likelihoods:")
if len(disagreement_df) > 0:
    counts = disagreement_df['symbol'].value_counts()
    for sym in counts.index:
        count = counts[sym]
        sym_df = disagreement_df[disagreement_df['symbol'] == sym]
        if len(sym_df) > 0:
            b_ll = sym_df['baseline_ll'].iloc[0]
            n_ll = sym_df['nh_ll'].iloc[0]
            print(f"{sym}: {count} rows | Base LL: {b_ll:.2f} | NH LL: {n_ll:.2f}")
else:
    print("WARNING: No disagreements found matching criteria!")


Rows after A_event_norm >= 0.01: 500
Stocks dropped due to <1 rows: []
Rows after dropping sparse stocks: 500
Capped at 15 rows per stock max.
Final disagreement count per stock and Likelihoods:
ASIANPAINT.NS: 15 rows | Base LL: 12796.28 | NH LL: 13200.82
DRREDDY.NS: 15 rows | Base LL: 12852.53 | NH LL: 13318.45
HINDUNILVR.NS: 15 rows | Base LL: 13170.28 | NH LL: 13639.26
ICICIBANK.NS: 15 rows | Base LL: 12555.33 | NH LL: 12903.09
INFY.NS: 15 rows | Base LL: 12595.59 | NH LL: 12899.55
LT.NS: 15 rows | Base LL: 12555.76 | NH LL: 12936.38
MARUTI.NS: 15 rows | Base LL: 12537.05 | NH LL: 12784.10
NESTLEIND.NS: 15 rows | Base LL: 13284.47 | NH LL: 13665.55
ONGC.NS: 15 rows | Base LL: 11783.37 | NH LL: 12009.99
RELIANCE.NS: 15 rows | Base LL: 12618.95 | NH LL: 12947.30
SUNPHARMA.NS: 15 rows | Base LL: 12789.97 | NH LL: 13127.24
WIPRO.NS: 15 rows | Base LL: 12557.69 | NH LL: 12829.31
KOTAKBANK.NS: 11 rows | Base LL: 12646.87 | NH LL: 13041.78
TCS.NS: 10 rows | Base LL: 12956.45 | NH LL: 13311

## 4. Save Disagreement Table


In [5]:
if len(disagreement_df) > 0:
    disagreement_df.to_parquet('../data/disagreement_table.parquet')
    disagreement_df.to_csv('../data/disagreement_table_stage1_REAL.csv', index=False)
    print("Saved disagreement_table.parquet and disagreement_table_stage1_REAL.csv to /data/")
else:
    # Create empty dataframe with schema just in case
    empty_df = pd.DataFrame(columns=[
        'symbol', 'event_date', 'event_type', 'baseline_state', 
        'baseline_label', 'nh_state', 'nh_label', 'baseline_ll', 
        'nh_ll', 'A_event_norm', 'days_to_event_signed'
    ])
    empty_df.to_parquet('../data/disagreement_table.parquet')
    print("Saved empty disagreement_table.parquet to /data/")


Saved disagreement_table.parquet and disagreement_table_stage1_REAL.csv to /data/
